In [1]:

!curl -fsSL https://ollama.com/install.sh | sh


import subprocess
import time

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

time.sleep(10)


!ollama pull llama3.1:8b


!ollama list


MODEL_NAME = "llama3.1:8b"
API_URL = "http://localhost:11434/api/generate"


>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.

NAME           ID              SIZE      MODIFIED               
llama3.1:8b    46e0c10c039e    4.9 GB    Less than a second ago    


In [2]:
import subprocess
import time

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

time.sleep(10)


In [6]:
# Install required packages
!pip install flask pyngrok requests

from flask import Flask, request, jsonify
from pyngrok import ngrok
import threading
import requests
import json

# Initialize Flask app
app = Flask(__name__)

# Your existing variables
MODEL_NAME = "llama3.1:8b"
API_URL = "http://localhost:11434/api/generate"

# Function to query Ollama
def query_ollama(user_input, context=""):
    """
    Query the Ollama model with user input and optional RAG context
    """
    prompt = f"{context}\n\nUser Question: {user_input}" if context else user_input

    payload = {
        "model": MODEL_NAME,
        "prompt": prompt,
        "stream": False
    }

    try:
        response = requests.post(API_URL, json=payload, timeout=120)
        response.raise_for_status()
        result = response.json()
        return result.get('response', 'No response generated')
    except Exception as e:
        return f"Error querying Ollama: {str(e)}"

# Flask endpoint
@app.route('/query', methods=['POST'])
def query():
    """
    Endpoint to receive user queries and return Ollama responses
    Expected JSON: {"query": "user question here", "context": "optional RAG context"}
    """
    try:
        data = request.get_json()
        user_input = data.get('query', '')
        rag_context = data.get('context', '')  # You'll add RAG retrieval here later

        if not user_input:
            return jsonify({'error': 'No query provided'}), 400

        # Query Ollama (add your RAG retrieval logic before this)
        response = query_ollama(user_input, rag_context)

        return jsonify({
            'response': response,
            'status': 'success'
        })
    except Exception as e:
        return jsonify({
            'error': str(e),
            'status': 'error'
        }), 500

@app.route('/health', methods=['GET'])
def health():
    """Health check endpoint"""
    return jsonify({'status': 'healthy', 'model': MODEL_NAME})

# Set up ngrok authentication (you'll need to sign up at ngrok.com for a free token)
# Get your token from: https://dashboard.ngrok.com/get-started/your-authtoken
from google.colab import userdata
ngrok.set_auth_token(userdata.get('ngrok_token'))  # Replace with your actual token

# Start ngrok tunnel
public_url = ngrok.connect(5000)
print("=" * 50)
print(f"🚀 PUBLIC URL: {public_url}")
print("=" * 50)
print(f"Use this URL in your local Flask app: {public_url}/query")
print("=" * 50)

# Run Flask app in a thread
def run_flask():
    app.run(port=5000, debug=False, use_reloader=False)

flask_thread = threading.Thread(target=run_flask)
flask_thread.daemon = True
flask_thread.start()

print("✅ Flask server is running!")
print("✅ Ngrok tunnel is active!")
print("\nTest the endpoint with:")
print(f'curl -X POST {public_url}/query -H "Content-Type: application/json" -d \'{{"query": "What is AI?"}}\'')

# Keep the cell running
import time
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("Server stopped")

🚀 PUBLIC URL: NgrokTunnel: "https://undaggled-nonrustically-eusebio.ngrok-free.dev" -> "http://localhost:5000"
Use this URL in your local Flask app: NgrokTunnel: "https://undaggled-nonrustically-eusebio.ngrok-free.dev" -> "http://localhost:5000"/query
✅ Flask server is running!
✅ Ngrok tunnel is active!

Test the endpoint with:
curl -X POST NgrokTunnel: "https://undaggled-nonrustically-eusebio.ngrok-free.dev" -> "http://localhost:5000"/query -H "Content-Type: application/json" -d '{"query": "What is AI?"}'
 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.


Server stopped


In [4]:
# # Install required packages
# # !pip install nano-graphrag ollama nest-asyncio networkx

# # !ollama serve
# subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
# import time
# time.sleep(5)

# # Imports
# import os
# import sys
# import gzip
# import json
# from pathlib import Path
# import nest_asyncio
# nest_asyncio.apply()

# import logging
# import ollama
# import numpy as np
# from nano_graphrag import GraphRAG, QueryParam
# from nano_graphrag.base import BaseKVStorage
# from nano_graphrag._utils import compute_args_hash, wrap_embedding_func_with_attrs
# from time import time
# import threading

# logging.basicConfig(level=logging.WARNING)
# logging.getLogger("nano-graphrag").setLevel(logging.INFO)

# # Model settings (matching your setup)
# MODEL = "llama3.1:8b"
# EMBEDDING_MODEL = "nomic-embed-text"
# EMBEDDING_MODEL_DIM = 768
# EMBEDDING_MODEL_MAX_TOKENS = 8192

# # Directories
# DATA_DIR = "./"  # ← CHANGE THIS to where your .json.gz files are
# N_RAGS = 3  # Number of parallel RAGs to create

# # LLM function with caching
# async def ollama_model_if_cache(
#     prompt, system_prompt=None, history_messages=[], **kwargs
# ) -> str:
#     kwargs.pop("max_tokens", None)
#     kwargs.pop("response_format", None)

#     ollama_client = ollama.AsyncClient()
#     messages = []
#     if system_prompt:
#         messages.append({"role": "system", "content": system_prompt})

#     hashing_kv: BaseKVStorage = kwargs.pop("hashing_kv", None)
#     messages.extend(history_messages)
#     messages.append({"role": "user", "content": prompt})

#     if hashing_kv is not None:
#         args_hash = compute_args_hash(MODEL, messages)
#         if_cache_return = await hashing_kv.get_by_id(args_hash)
#         if if_cache_return is not None:
#             return if_cache_return["return"]

#     response = await ollama_client.chat(model=MODEL, messages=messages, **kwargs)
#     result = response["message"]["content"]

#     if hashing_kv is not None:
#         await hashing_kv.upsert({args_hash: {"return": result, "model": MODEL}})

#     return result

# # Embedding function
# @wrap_embedding_func_with_attrs(
#     embedding_dim=EMBEDDING_MODEL_DIM,
#     max_token_size=EMBEDDING_MODEL_MAX_TOKENS,
# )
# async def ollama_embedding(texts: list[str]) -> np.ndarray:
#     embed_text = []
#     for text in texts:
#         data = ollama.embeddings(model=EMBEDDING_MODEL, prompt=text)
#         embed_text.append(data["embedding"])
#     return embed_text

# # Pull embedding model if not already available
# !ollama pull nomic-embed-text

# # Create data directory if it doesn't exist
# !mkdir -p {DATA_DIR}

# print("Upload your .json.gz files to the data directory")
# print(f"Current working directory: {os.getcwd()}")
# print(f"Data directory: {DATA_DIR}")


# def create_single_rag_from_items(rag_idx, items_to_process, results_dict):
#     """Create a RAG from a list of text items"""
#     working_dir = f"./nano_graphrag_cache_rag_{rag_idx}"

#     print(f"\n🚀 Starting RAG {rag_idx + 1} (Thread {threading.current_thread().name}) with {len(items_to_process)} items")

#     try:
#         # Clear previous cache
#         for file in [
#             f"{working_dir}/vdb_entities.json",
#             f"{working_dir}/kv_store_full_docs.json",
#             f"{working_dir}/kv_store_text_chunks.json",
#             f"{working_dir}/kv_store_community_reports.json",
#             f"{working_dir}/graph_chunk_entity_relation.graphml"
#         ]:
#             if os.path.exists(file):
#                 os.remove(file)

#         rag = GraphRAG(
#             working_dir=working_dir,
#             enable_llm_cache=True,
#             best_model_func=ollama_model_if_cache,
#             cheap_model_func=ollama_model_if_cache,
#             embedding_func=ollama_embedding,
#             addon_params={
#                 "enable_clustering": False,
#             },
#         )

#         start = time()

#         # Insert all items
#         for i, text in enumerate(items_to_process):
#             if (i + 1) % 10 == 0 or i == 0 or i == len(items_to_process) - 1:
#                 print(f"[RAG {rag_idx + 1}] Progress: {i+1}/{len(items_to_process)} items")
#             rag.insert(text)

#         elapsed = time() - start
#         print(f"✅ RAG {rag_idx + 1} completed in {elapsed:.2f}s")
#         results_dict[rag_idx] = {"status": "success", "time": elapsed, "dir": working_dir}

#     except Exception as e:
#         print(f"❌ RAG {rag_idx + 1} failed: {str(e)}")
#         import traceback
#         traceback.print_exc()
#         results_dict[rag_idx] = {"status": "failed", "error": str(e)}


# def insert_parallel_by_items():
#     """Split items from files across multiple RAGs"""

#     # Get all .json.gz files
#     gz_files = list(Path(DATA_DIR).glob("*.json.gz"))
#     print(f"Found {len(gz_files)} .json.gz files")

#     if len(gz_files) == 0:
#         print("❌ No .json.gz files found!")
#         return []

#     # Load ALL items from ALL files
#     all_items = []
#     for file in gz_files:
#         print(f"Loading: {file}")
#         with gzip.open(file, "rt", encoding="utf-8") as f:
#             obj = json.load(f)

#         if isinstance(obj, list):
#             for item in obj:
#                 text = item.get("text", "").strip()
#                 if text:
#                     all_items.append(text)
#                     print(f"  Loaded item {len(all_items)}: {len(text)} chars")
#         elif isinstance(obj, dict):
#             text = obj.get("text", "").strip()
#             if text:
#                 all_items.append(text)
#                 print(f"  Loaded item {len(all_items)}: {len(text)} chars")

#     print(f"\nTotal items to process: {len(all_items)}")

#     if len(all_items) == 0:
#         print("❌ No text content found!")
#         return []

#     # Split items across N RAGs (round-robin)
#     items_per_rag = [[] for _ in range(N_RAGS)]
#     for i, item in enumerate(all_items):
#         items_per_rag[i % N_RAGS].append(item)

#     # Print distribution
#     print(f"\n{'='*60}")
#     print(f"Splitting {len(all_items)} items across {N_RAGS} RAGs:")
#     for i, items in enumerate(items_per_rag):
#         total_chars = sum(len(item) for item in items)
#         print(f"  RAG {i + 1}: {len(items)} items ({total_chars:,} total chars)")
#     print(f"{'='*60}\n")

#     # Results dictionary (thread-safe)
#     results = {}
#     threads = []

#     print(f"🚀 Starting {N_RAGS} RAGs in parallel...")
#     overall_start = time()

#     for i in range(N_RAGS):
#         thread = threading.Thread(
#             target=create_single_rag_from_items,
#             args=(i, items_per_rag[i], results),
#             name=f"RAG-{i+1}"
#         )
#         threads.append(thread)
#         thread.start()

#     # Wait for all threads to complete
#     for thread in threads:
#         thread.join()

#     overall_time = time() - overall_start

#     # Print summary
#     print(f"\n{'='*60}")
#     print(f"📊 PARALLEL TRAINING SUMMARY")
#     print(f"{'='*60}")
#     print(f"Total parallel time: {overall_time:.2f}s")

#     successful = [r for r in results.values() if r.get("status") == "success"]
#     failed = [r for r in results.values() if r.get("status") == "failed"]

#     if successful:
#         avg_time = sum(r["time"] for r in successful) / len(successful)
#         sequential_estimate = sum(r["time"] for r in successful)
#         speedup = sequential_estimate / overall_time if overall_time > 0 else 1

#         print(f"Successful RAGs: {len(successful)}/{N_RAGS}")
#         print(f"Average time per RAG: {avg_time:.2f}s")
#         print(f"Estimated sequential time: {sequential_estimate:.2f}s")
#         print(f"Speedup: {speedup:.2f}x")
#         print()

#     for i in range(N_RAGS):
#         result = results.get(i, {})
#         if result.get("status") == "success":
#             print(f"  ✅ RAG {i + 1}: {result['time']:.2f}s - {result['dir']}")
#         else:
#             print(f"  ❌ RAG {i + 1}: {result.get('error', 'unknown error')}")

#     print(f"{'='*60}\n")

#     return [results[i]["dir"] for i in range(N_RAGS) if results.get(i, {}).get("status") == "success"]


# # Run parallel insertion
# rag_directories = insert_parallel_by_items()

# if len(rag_directories) == 0:
#     print("❌ No RAGs were created successfully. Please check the errors above.")
# else:
#     # Package all RAGs for download
#     import shutil
#     from google.colab import files

#     print("📦 Creating zip files for download...")
#     for i, rag_dir in enumerate(rag_directories):
#         zip_name = f'nano_graphrag_cache_rag_{i}'
#         print(f"  Zipping {rag_dir}...")
#         shutil.make_archive(zip_name, 'zip', rag_dir)
#         print(f"  ✅ Created {zip_name}.zip")

#     print(f"\n⬇️ Downloading {len(rag_directories)} RAG cache files...")
#     for i in range(len(rag_directories)):
#         print(f"  Downloading RAG {i}...")
#         files.download(f'nano_graphrag_cache_rag_{i}.zip')

#     print("\n✅ All downloads complete!")
#     print(f"Extract these {len(rag_directories)} zip files on your local machine.")
#     print("\nTo query locally, extract all RAG folders and use the federated query code provided earlier.")